In [3]:
import os
from dotenv import load_dotenv
import warnings

# 경고 메시지를 무시하도록 설정합니다.
warnings.filterwarnings('ignore')

# 현재 작업 폴더에 있는 .env파일을 읽습니다.
# .env 안에 OPEN_API_KEY=...라고 적어두면,
# os.environ["OPEN_API_KEY"] 형태로 사용할 수 있게 됩니다.
load_dotenv()

# API Key가 제대로 불러와졌는지 확인합니다.
# 보안상 전체 키를 출력하면 안 되므로, 존재 여부만 확인합니다.
print("OPENAI_API_KEY 로드 여부:", "OPENAI_API_KEY" in os.environ)

OPENAI_API_KEY 로드 여부: True


#### 1번 가장 기본적인 Tool Agent 만들기
##### Agent = LLM + Tool + 판단 로직

사용자 질문
→ LLM이 질문을 이해함
→ 필요한 Tool이 있는지 판단함
→ Tool 실행
→ Tool 결과를 보고 최종 답변 생성

In [4]:
## LangChain에서 Agent를 생성하는 함수 입니다.
# Agent는 LLM이 도구를 사용할 수 있게 만들어주는 실행구조입니다.
from langchain.agents import create_agent
# OpenAI의 ChatGPT 계열 모델을 LangChain에서 사용할 수 있게 해주는 클래스입니다.
from langchain_openai import ChatOpenAI

# -------------------------------------------------
# 1. Tool로 사용할 일반 파이썬 함수 정의
# -------------------------------------------------
def get_weather(city: str) -> str:
    """
    주어진 도시의 날씨를 알려주는 함수입니다.

    이 함수는 실제 날씨 API를 호출하는 함수는 아닙니다.
    실습용으로 도시이름에 따라 정해진 문자열을 반환합니다.

    Args:
        city (str): 사용자가 물어본 도시 이름

    Returns:
        str: 해당 도시의 날씨 정보
    """

    # 사용자가 입력한 city 문자열 안에 "서울"이 포함되어 있으면
    # 서울 날씨를 반환합니다.
    if "서울" in city:
        return f"{city}의 날씨는 비가 오고 있습니다."

    # 서울이 아닌 다른 도시는 모두 맑다고 가정합니다.
    return f"It's always sunny in {city}!"

In [5]:
# -------------------------------------------------
# 2. LLM 모델 설정
# -------------------------------------------------
llm = ChatOpenAI(
    # 사용할 OpenAI 모델 이름입니다.
    # 실습에서는 비용이 상대적으로 낮고 빠른 gpt-40-mini를 추천합니다.
    model='gpt-4o-mini',

    # .env파일에서 불러온 OpenAI API KEY를 사용합니다.
    # 기존 강의 코드의 PROXY_TOKEN 대신 OPEN_API_KEY를 사용합니다.
    api_key=os.environ['OPENAI_API_KEY'],
    
    # temperature는 답변의 창의성/랜덤성을 조절합니다.
    # 0에 가까울수록 같은 질문에 대해 더 일관된 답변을 합니다. 
    temperature=0,
)

# -------------------------------------------------
# 3. Agent 생성
# -------------------------------------------------
agent = create_agent(
    # Agent가 사용할 LLM입니다.
    model=llm,

    # Agent가 사용할 수 있는 도구 목록입니다.
    # 여기서는 get_weather 함수 하나만 도구로 제공합니다.
    tools=[get_weather],

    # 시스템 프롬프트입니다.
    # Agent의 역할이나 말투, 규칙을 지정합니다.
    system_prompt='You are a helpful assistant',
)

# -------------------------------------------------
# 4. Agent 실행
# -------------------------------------------------
res = agent.invoke(
    {
        # LangChain Agent는 대화기록을 messages 형태로 받습니다.
        # role은 user, assistant, system등이 들어갈 수 있습니다.
        'messages': [
            {
                "role": "user",
                "content": "서울의 날씨는 어때?"
            }
        ]
    }
)

# -------------------------------------------------
# 5. 결과 확인
# -------------------------------------------------

# res에는 Agent가 처리한 전체 메시지 흐름이 들어있습니다. 
# 보통 사용자 메시지, 도구 호출 메시지, 최종 답변 메시지 등이 포함됩니다. 
for msg in res['messages']:
    msg.pretty_print()

================================ Human Message =================================

서울의 날씨는 어때?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_Vli36Ngbw13OhdP0n2ViBwMK)
 Call ID: call_Vli36Ngbw13OhdP0n2ViBwMK
  Args:
    city: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 비가 오고 있습니다.
================================== Ai Message ==================================

서울의 날씨는 비가 오고 있습니다.


#### 2번 시스템 프롬프트 정의
Agent의 행동 규칙을 정하는 부분

##### get_weather_for_location(city)는 사용자가 도시를 직접 말했을 때 쓰는 도구입니다.
##### 반면에 get_user_location(runtime)은 사용자가 도시를 말하지 않았을 때 필요합니다.

##### 만약 사용자가 도시를 말하지않았을 경우: 사용자가 현재 위치의 날씨를 묻는 것 같네.
그런데 위치를 모르네.
get_user_location 도구로 위치를 먼저 알아내자.
그 다음 get_weather_for_location 도구로 날씨를 확인하자.

In [6]:
SYSTEM_PROMPT = """
당신은 언어유희(말장난)를 섞어 말하는 전문 기상 캐스터입니다.

당신은 다음 두 가지 도구에 접근할 수 있습니다:

- get_weather_for_location: 특정 지역의 날씨 정보를 가져올 때 사용하세요.
- get_user_location: 사용자의 위치 정보를 가져올 때 사용하세요.

사용자가 날씨를 물어볼 경우, 반드시 대상 위치를 확인해야 합니다.

만약 사용자의 질문 문맥상 현재 자신이 있는 곳의 날씨를 묻는 것이라고 판단된다면,
get_user_location 도구를 사용하여 사용자의 위치를 찾으세요.

답변은 한국어로 해주세요.
"""

In [7]:
# dataclass는 데이터를 담는 간단한 클래스를 만들때 사용합니다.
# 여기서는 사용자 정보를 담는 Context 클래스를 만들기위해 사용합니다.
from dataclasses import dataclass

# tool은 일반 함수를 LangChain Tool로 등록하기위한 데코레이터입니다.
# ToolRuntime은 Agent 실행중 전달되는 context 정보에 접근할때 사용합니다.
from langchain.tools import tool, ToolRuntime

# -------------------------------------------------
# 1. 특정 지역의 날씨를 알려주는 Tool
# -------------------------------------------------
@tool
def get_weather_for_location(city: str) -> str:
    """
    특정 도시의 날씨정보를 반환합니다.

    Args:
        city (str): 날씨를 알고 싶은 도시 이름

    Returns:
        str: 해당 도시의 날씨 정보
    """

    # 실제 서비스라면 여기에서 날씨 API를 호출할 수 있습니다.
    # 지금은 실습이므로 항상 맑다고 가정합니다.
    return f"{city}의 날씨는 항상 맑습니다."

# -------------------------------------------------
# 2. Runtime Context 정의
# -------------------------------------------------
@dataclass
class Context:
    """
    Agent 실행 중 함께 전달할 사용자 정보를 담는 클래스입니다.

    예를 들어, 로그인한 사용자 ID, 사용자 위치, 사용자 권한 등
    LLM에게 직접 노출하거나 Tool에서 참고할 정보를 넣을수 있습니다.
    """

    # 현재 사용자의 ID입니다.
    user_id: str


# -------------------------------------------------
# 3. 사용자의 위치를 가져오는 Tool
# -------------------------------------------------
@tool
def get_user_location(runtime: ToolRuntime[Context]) -> str:
    """
    사용자 ID를 기반으로 사용자의 위치 정보를 반환합니다.

    이 Tool은 사용자가 직접 도시를 말하지않고
    '밖에 날씨 어때?' 처럼 물어볼때 사용할 수 있습니다.

    Args:
        runtime (ToolRuntime[Context]):
            Agent 실행 중 전달된 Context에 접근할 수 있는 객체

    Returns:
        str: 사용자의 현재 위치
    """

    # runtime.context 안에 우리가 전달한 Context 객체가 들어 있습니다.
    user_id = runtime.context.user_id

    # 실습용 예시입니다.
    # user_id가 "1"이면 서울, 아니면 부산이라고 가정합니다.
    if user_id == '1':
        return '서울'
    else:
        return '부산'

In [8]:
# LangChain의 모델 초기화 함수입니다.
# 다만 태영님 입장에서는 ChatOpenAI를 직접 쓰는 방식이 더 직관적입니다.
from langchain_openai import ChatOpenAI

model = ChatOpenAI(
    # 사용할 모델입니다.
    model='gpt-4o-mini',

    # .env에서 불러온 OpenAI API Key입니다.
    api_key=os.environ["OPENAI_API_KEY"],

    # 답변의 랜덤성을 낮춥니다.
    temperature=0,
)

In [9]:
# 구조화된 출력을 하기위해 2가지로 출력하게 설정하는코드
from dataclasses import dataclass

@dataclass
class ResponseFormat:
    """
    Agent가 최종적으로 반환해야하는 응답 구조입니다.

    이 클래스를 정의하면 Agent가 답변을 아무렇게나 하지 않고,
    아래 필드에 맞춰서 결과를 만들어줍니다.
    """

    # 말장난이 섞인 최종 답변입니다.
    # 필수값이므로 기본값을 주지 않습니다.
    punny_response: str

    # 날씨 정보입니다.
    # 날씨 질문이 아닌 경우에는 None이 될 수 있습니다.
    weather_conditions: str | None = None

In [10]:
# Agent가 이전 대화를 기억하게 하려면 체크포인터가 필요함
# 중요한 것은 thread_id를 통해 기억을 하는것이 아래 메모리의 특징임
# LangGraph에서 제공하는 메모리 저장소입니다.
# 실행 중 대화 상태를 메모리에 저장합니다.
from langgraph.checkpoint.memory import InMemorySaver

# 메모리 저장 객체 생성
checkpointer = InMemorySaver()

In [11]:
# 최종 Agent 구성
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy

# -------------------------------------------------
# 최종 Agent 생성
# -------------------------------------------------
agent = create_agent(
    # 사용할 LLM 모델입니다.
    model=model,

    # Agent의 역할과 도구 사용 규칙입니다.
    system_prompt=SYSTEM_PROMPT,

    # Agent가 사용할수 있는 도구 목록입니다.
    # 사용자의 위치확인 도구와 날씨 조회 도구를 모두 넣습니다.
    tools=[
        get_user_location,
        get_weather_for_location
    ],

    # ToolRuntime에서 사용할 Context 구조를 지정합니다.
    # 이렇게 해야 get_user_location에서 runtime.context.user_id를 사용할 수 있습니다.
    context_schema=Context,

    # Agent의 최종 응답을 ResponseFormat 구조에 맞게 받습니다.
    response_format=ToolStrategy(ResponseFormat),

    # 대화 기억을 위한 체크포인터입니다.
    checkpointer=checkpointer,
)


# -------------------------------------------------
# 같은 대화방을 구분하기 위한 thread_id 설정
# -------------------------------------------------

config = {
    "configurable": {
        # 같은 thread_id를 사용하면 이전 대화를 기억합니다.
        "thread_id": "1"
    }
}

# -------------------------------------------------
# Agent 실행 1: 위치를 말하지 않고 날씨 질문하기
# -------------------------------------------------

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "밖에 날씨 어때?"
            }
        ]
    },

    # 대화 세션 정보입니다.
    config=config,

    # 사용자 정보를 Context로 전달합니다.
    # user_id가 "1"이므로 get_user_location Tool은 "서울"을 반환합니다.
    context=Context(user_id="1")
)


# -------------------------------------------------
# 구조화된 응답 출력
# -------------------------------------------------

print(response["structured_response"])

print("말장난 답변:", response["structured_response"].punny_response)
print("날씨 정보:", response["structured_response"].weather_conditions)


ResponseFormat(punny_response='서울의 날씨는 항상 맑습니다! 맑은 날씨에 기분도 맑아지겠네요!', weather_conditions='맑음')
말장난 답변: 서울의 날씨는 항상 맑습니다! 맑은 날씨에 기분도 맑아지겠네요!
날씨 정보: 맑음


# LangChain Agent 구조화 출력(ResponseFormat) 쉽게 이해하기

## 1. 헷갈리는 코드

```python
from dataclasses import dataclass

@dataclass
class ResponseFormat:
    punny_response: str
    weather_conditions: str | None = None
```

처음 보면 이런 의문이 생길 수 있습니다.

> `punny_response`를 재미있게 만드는 함수도 없고,  
> `weather_conditions`에 날씨 정보를 넣는 함수도 없는데  
> 어떻게 LLM이 알아서 값을 채우는 거지?

이 질문은 아주 중요합니다.

결론부터 말하면, `ResponseFormat`은 값을 계산하는 함수가 아닙니다.

---

## 2. ResponseFormat은 “답변을 담는 그릇”이다

`ResponseFormat`은 최종 답변의 모양을 정하는 코드입니다.

즉, 이런 뜻입니다.

```text
최종 답변은 반드시 아래 두 칸으로 나눠서 줘.

1. punny_response: 문자열
2. weather_conditions: 문자열 또는 None
```

다시 말해 `ResponseFormat`은 다음 역할을 합니다.

```text
무엇을 답할지 결정하는 코드 X
어떤 모양으로 답할지 정하는 코드 O
```

---

## 3. 엑셀 양식으로 비유하기

`ResponseFormat`은 엑셀 양식과 비슷합니다.

```text
| punny_response | weather_conditions |
|----------------|--------------------|
|                |                    |
```

이 양식만 만든다고 자동으로 값이 생기지는 않습니다.

하지만 LLM에게 이렇게 지시하는 효과가 있습니다.

```text
너 답변할 때 자유롭게 긴 문장으로만 답하지 말고,
반드시 이 표의 칸에 맞춰서 답해.
```

즉, `ResponseFormat`은 답변 내용을 만드는 코드가 아니라,  
LLM이 만든 답변을 정해진 칸에 나눠 담게 만드는 코드입니다.

---

## 4. 그러면 실제 값은 누가 만드나?

실제 값을 만드는 것은 파이썬 함수가 아니라 LLM입니다.

LLM은 아래 정보를 종합해서 최종 답변을 만듭니다.

```text
1. 사용자의 질문
2. system_prompt
3. 사용 가능한 tool
4. tool 실행 결과
5. ResponseFormat의 필드 이름
6. ResponseFormat의 필드 타입
7. 클래스 설명 또는 Field 설명
```

---

## 5. 예시로 이해하기

사용자가 이렇게 물어봤다고 가정합니다.

```text
밖에 날씨 어때?
```

Agent는 도구를 실행해서 이런 결과를 얻습니다.

```text
서울의 날씨는 항상 맑습니다!
```

그리고 시스템 프롬프트에는 이런 내용이 들어 있습니다.

```python
SYSTEM_PROMPT = '''
당신은 언어유희(말장난)를 섞어 말하는 전문 기상 캐스터입니다.
사용자가 날씨를 물어보면 위치를 확인하고 날씨 도구를 사용하세요.
답변은 한국어로 해주세요.
'''
```

그러면 LLM은 이렇게 판단합니다.

```text
사용자가 날씨를 물어봤네.
도구 결과는 "서울의 날씨는 항상 맑습니다!"네.
시스템 프롬프트에서 말장난을 섞으라고 했네.
최종 출력 형식은 ResponseFormat이네.

그러면:
punny_response에는 말장난이 섞인 최종 답변을 넣고,
weather_conditions에는 도구로 확인한 날씨 정보를 넣자.
```

그래서 최종적으로 이런 구조가 만들어집니다.

```python
ResponseFormat(
    punny_response="서울은 오늘도 해가 쨍쨍하네요. 기분도 같이 맑음으로 갱신해보세요!",
    weather_conditions="서울의 날씨는 항상 맑습니다!"
)
```

---

## 6. 중요한 핵심

여기서 가장 중요한 점은 이것입니다.

```text
weather_conditions 값을 넣는 함수가 따로 있는 것이 아니다.
LLM이 도구 결과를 보고 weather_conditions 필드에 맞게 채우는 것이다.
```

그리고 `punny_response`도 마찬가지입니다.

```text
punny_response를 만드는 함수가 따로 있는 것이 아니다.
LLM이 system_prompt를 보고 말장난이 섞인 답변을 직접 작성하는 것이다.
```

---

## 7. 필드 이름이 중요한 이유

LLM은 필드 이름을 보고 의미를 추측합니다.

좋은 예:

```python
@dataclass
class ResponseFormat:
    punny_response: str
    weather_conditions: str | None = None
```

이름이 명확합니다.

```text
punny_response → 말장난이 섞인 응답
weather_conditions → 날씨 상태
```

나쁜 예:

```python
@dataclass
class ResponseFormat:
    x: str
    y: str | None = None
```

이 경우 LLM 입장에서는 `x`, `y`가 무엇을 의미하는지 알기 어렵습니다.

그래서 구조화 출력에서는 필드 이름을 의미 있게 짓는 것이 매우 중요합니다.

---

## 8. 타입의 의미

```python
punny_response: str
```

이 뜻은 다음과 같습니다.

```text
punny_response에는 문자열이 들어가야 한다.
```

예:

```python
punny_response = "서울은 오늘도 햇살 맛집입니다!"
```

---

```python
weather_conditions: str | None = None
```

이 뜻은 다음과 같습니다.

```text
weather_conditions에는 문자열이 들어가도 되고,
날씨 정보가 없으면 None이 들어가도 된다.
```

예:

```python
weather_conditions = "서울의 날씨는 항상 맑습니다!"
```

또는 날씨 질문이 아니라면:

```python
weather_conditions = None
```

---

## 9. `= None`의 의미

```python
weather_conditions: str | None = None
```

여기서 `= None`은 기본값입니다.

즉, 날씨 정보가 없으면 비워둘 수 있다는 뜻입니다.

반면에 아래 필드는 기본값이 없습니다.

```python
punny_response: str
```

그래서 `punny_response`는 반드시 채워져야 합니다.

정리하면 다음과 같습니다.

```text
punny_response → 필수값
weather_conditions → 선택값, 없으면 None 가능
```

---

## 10. 일반 파이썬 함수와 LLM 구조화 출력의 차이

### 일반 파이썬 함수

일반 파이썬에서는 개발자가 직접 값을 만드는 로직을 작성해야 합니다.

```python
def make_punny_response(weather):
    return f"{weather}네요! 기분도 맑게 가요!"

def make_weather_conditions(city):
    return f"{city}의 날씨는 항상 맑습니다!"
```

이 경우 값은 함수가 만듭니다.

```text
개발자가 작성한 함수 로직이 값을 만든다.
```

---

### LLM 구조화 출력

반면 LLM 구조화 출력에서는 개발자가 답변 양식만 정합니다.

```python
@dataclass
class ResponseFormat:
    punny_response: str
    weather_conditions: str | None = None
```

실제 내용은 LLM이 만듭니다.

```text
개발자는 답변 양식만 정한다.
LLM이 프롬프트와 도구 결과를 보고 내용을 채운다.
```

---

## 11. 왜 pydantic Field를 쓰면 더 좋은가?

`dataclass` 방식은 간단하지만, 필드별 설명을 자세히 전달하기 어렵습니다.

```python
@dataclass
class ResponseFormat:
    punny_response: str
    weather_conditions: str | None = None
```

LLM 입장에서는 대략 이렇게만 이해합니다.

```text
punny_response라는 칸이 있네.
weather_conditions라는 칸이 있네.
```

하지만 `pydantic`의 `Field(description=...)`를 사용하면 각 필드의 의미를 더 명확하게 전달할 수 있습니다.

```python
from pydantic import BaseModel, Field

class ResponseFormat(BaseModel):
    punny_response: str = Field(
        description="사용자에게 보여줄 최종 답변입니다. 반드시 말장난이나 언어유희를 포함해야 합니다."
    )

    weather_conditions: str | None = Field(
        default=None,
        description="도구로 확인한 실제 날씨 정보입니다. 날씨 질문이 아닌 경우 None으로 둡니다."
    )
```

이렇게 하면 LLM은 훨씬 명확하게 이해합니다.

```text
punny_response에는 말장난이 포함된 최종 답변을 넣어야 하는구나.
weather_conditions에는 도구로 확인한 실제 날씨 정보를 넣어야 하는구나.
```

---

## 12. 전체 흐름 그림

```text
사용자 질문
   ↓
"밖에 날씨 어때?"

SYSTEM_PROMPT
   ↓
"너는 말장난하는 기상 캐스터야"

Tool 실행
   ↓
get_user_location → "서울"
get_weather_for_location("서울") → "서울의 날씨는 항상 맑습니다!"

ResponseFormat
   ↓
최종 답변은 이 모양으로 만들어:
{
  "punny_response": 문자열,
  "weather_conditions": 문자열 또는 None
}

LLM 최종 작성
   ↓
{
  "punny_response": "서울은 오늘도 햇살이 열일 중입니다!",
  "weather_conditions": "서울의 날씨는 항상 맑습니다!"
}
```

---

## 13. 한 문장으로 정리

```text
ResponseFormat은 답변을 만드는 함수가 아니라,
LLM이 만든 답변을 어떤 칸에 나눠 담을지 정하는 설계도다.
```

그리고 실제 내용은 다음 자료를 보고 LLM이 채웁니다.

```text
system_prompt + 사용자 질문 + tool 결과 + 필드 이름 + 필드 타입
```

---

## 14. 공부할 때 기억할 표현

```text
ResponseFormat = 출력 양식
Field 이름 = LLM이 이해하는 칸 이름
Field 타입 = 각 칸에 들어갈 값의 종류
system_prompt = LLM의 행동 규칙
tool 결과 = LLM이 참고할 실제 정보
LLM = 양식에 맞게 내용을 채우는 작성자
```

---

## 15. 최종 추천 코드

공부용으로는 `dataclass`도 괜찮습니다.

```python
from dataclasses import dataclass

@dataclass
class ResponseFormat:
    punny_response: str
    weather_conditions: str | None = None
```

하지만 더 명확하고 안정적인 구조화 출력을 원한다면 아래처럼 쓰는 것을 추천합니다.

```python
from pydantic import BaseModel, Field

class ResponseFormat(BaseModel):
    punny_response: str = Field(
        description="사용자에게 보여줄 최종 답변입니다. 반드시 말장난이나 언어유희를 포함합니다."
    )

    weather_conditions: str | None = Field(
        default=None,
        description="도구를 통해 확인한 실제 날씨 정보입니다. 날씨 질문이 아니면 None으로 둡니다."
    )
```

---

## 16. 제일 중요한 결론

태영님이 헷갈렸던 지점은 이것입니다.

```text
함수가 없는데 어떻게 punny_response랑 weather_conditions가 채워지지?
```

답은 다음과 같습니다.

```text
일반 파이썬 함수가 채우는 것이 아니다.
LLM이 프롬프트와 도구 결과를 보고 채운다.
```

그리고 `ResponseFormat`은 LLM에게 이렇게 말하는 코드입니다.

```text
답변을 아무렇게나 하지 말고,
반드시 이 구조에 맞춰서 만든다.
```


In [12]:
from langchain.tools import tool


@tool
def multiply_numbers(a: int, b: int) -> int:
    """
    두 정수 a와 b를 곱한 결과를 반환합니다.

    Args:
        a (int): 첫 번째 정수
        b (int): 두 번째 정수

    Returns:
        int: a와 b를 곱한 값
    """

    return a * b


print(f"Tool 이름: {multiply_numbers.name}")
print(f"설명: {multiply_numbers.description}")

Tool 이름: multiply_numbers
설명: 두 정수 a와 b를 곱한 결과를 반환합니다.

Args:
    a (int): 첫 번째 정수
    b (int): 두 번째 정수

Returns:
    int: a와 b를 곱한 값


In [13]:
# Runtime Context 활용
import os
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool, ToolRuntime


# .env에서 OPENAI_API_KEY 불러오기
load_dotenv()


@dataclass
class GameContext:
    """
    게임 사용자 정보를 담는 Context입니다.

    Agent 실행 시 context=GameContext(level="100")처럼 전달하면,
    Tool 내부에서 runtime.context.level로 접근할 수 있습니다.
    """

    level: str


@tool
def check_user_level(runtime: ToolRuntime[GameContext]) -> str:
    """
    Context에 저장된 사용자의 게임 레벨을 확인합니다.

    Args:
        runtime (ToolRuntime[GameContext]):
            Agent 실행 중 전달된 GameContext에 접근하는 객체

    Returns:
        str: 사용자의 현재 레벨 안내 문장
    """

    level = runtime.context.level
    return f"현재 레벨은 {level}입니다."


llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)


agent = create_agent(
    model=llm,

    # Agent가 사용할 수 있는 도구입니다.
    tools=[check_user_level],

    # ToolRuntime에서 사용할 Context 구조입니다.
    context_schema=GameContext,

    # Agent의 행동 규칙입니다.
    system_prompt=(
        "당신은 사용자를 서포트해주는 도우미입니다. "
        "사용자가 자신의 레벨을 물어보면 반드시 check_user_level 도구를 이용해서 답변해주세요."
    ),
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "내 레벨 알려줘"
            }
        ]
    },

    # 실행 시점에 사용자 레벨 정보를 전달합니다.
    context=GameContext(level="100"),
)


print("=== RAW RESULT ===")
print(result)


if isinstance(result, dict) and "messages" in result and result["messages"]:
    last_msg = result["messages"][-1]
    content = getattr(last_msg, "content", None)
    print("\n=== FINAL ANSWER ===")
    print(content)

=== RAW RESULT ===
{'messages': [HumanMessage(content='내 레벨 알려줘', additional_kwargs={}, response_metadata={}, id='570d3ede-e94a-47f9-a6d4-32dd54d1f685'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 11, 'prompt_tokens': 125, 'total_tokens': 136, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_4181e24c46', 'id': 'chatcmpl-DZGbfd90NUWFS5iOJ0V1Y2amNnf7M', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019dcf32-8199-7e50-9d91-1b5057a50fbf-0', tool_calls=[{'name': 'check_user_level', 'args': {}, 'id': 'call_08vma5HMBoNmk4pPF6Bs8Zkb', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 125, 'output_tokens': 1

In [14]:
# 구조화된 출력
import os
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


load_dotenv()


@dataclass
class AnalysisResult:
    """
    리뷰 분석 결과를 담는 구조입니다.

    summary에는 리뷰 요약,
    sentiment에는 감성 분석 결과가 들어갑니다.
    """

    summary: str
    sentiment: str


llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)


agent = create_agent(
    model=llm,
    tools=[],
    system_prompt=(
        "당신은 리뷰를 분석하는 분석가입니다. "
        "사용자의 리뷰를 요약하고, 감성을 긍정/부정/중립 중 하나로 분석하세요."
    ),

    # Agent의 최종 응답을 AnalysisResult 구조로 강제합니다.
    response_format=ToolStrategy(AnalysisResult),
)


test_input = {
    "messages": [
        {
            "role": "user",
            "content": "이 제품은 정말 최고예요! 사용하기도 편하고 디자인도 예뻐서 강력 추천합니다."
        }
    ]
}


response = agent.invoke(test_input)

structured_res = response["structured_response"]

print(f"결과 타입: {type(structured_res)}")
print(f"요약: {structured_res.summary}")
print(f"감성: {structured_res.sentiment}")

결과 타입: <class '__main__.AnalysisResult'>
요약: 제품이 사용하기 편하고 디자인이 예쁘며 강력 추천하는 리뷰입니다.
감성: 긍정


In [15]:
import os
from dataclasses import dataclass

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.agents.structured_output import ToolStrategy


load_dotenv()


@dataclass
class TravelRecommendation:
    """
    여행 추천 결과를 담는 구조입니다.

    destination: 추천 여행지
    reason: 추천 이유
    travel_type: 여행 유형
    """

    destination: str
    reason: str
    travel_type: str


llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0,
)


agent = create_agent(
    model=llm,
    tools=[],

    system_prompt="""
    당신은 국내외 여행지를 매우 잘 아는 여행 추천 전문가입니다.

    사용자의 예산, 기간, 여행 성향을 고려해 여행지를 추천하세요.
    답변은 반드시 현실적인 추천이어야 합니다.

    특히 사용자가 여행 초보라면,
    이동이 너무 복잡하지 않고 부담이 적은 여행지를 우선 추천하세요.
    """,

    response_format=ToolStrategy(TravelRecommendation),
)


test_input = {
    "messages": [
        {
            "role": "user",
            "content": "나는 국내 여행을 다니면서 조용하지만 평화로운 여행지를 찾고 있어. 예산은 50만원 정도고 1박 2일 정도 있으려고 해."
        }
    ]
}


response = agent.invoke(test_input)

structured_res = response["structured_response"]

print(f"추천 여행지: {structured_res.destination}")
print(f"추천 이유: {structured_res.reason}")
print(f"여행 유형: {structured_res.travel_type}")

추천 여행지: 강릉
추천 이유: 강릉은 아름다운 해변과 조용한 자연 경관이 어우러져 있어 평화로운 여행을 즐기기에 적합합니다. 또한, 맛있는 해산물과 커피도 즐길 수 있습니다.
여행 유형: 국내 여행
